# FusionMatch — Phase 1: Data Pipeline
### Multimodal (Image + Text) Product Deduplication Engine

This notebook orchestrates and validates **Phase 1 (Data Pipeline)** of FusionMatch:
1. **ABO Dataset Ingestion**: Downloads and extracts `abo-images-small.tar` (256px images) and `abo-listings.tar` (metadata).
2. **Metadata & Image Resolution**: Parses sharded JSONL listings, resolves image paths, extracts clean English text.
3. **Stratified Sampling**: Samples ~10,000–11,000 SKUs balanced across product categories.
4. **SKU-Level Splitting**: Partitions catalog into Train (80%), Val (10%), and Test (10%) with **zero SKU leakage**.
5. **Multimodal Augmentation & Pair Sampling**: Visualizes multi-angle positive pairs, augmented duplicates, and hard negatives.

In [ ]:
# Cell 1: Environment & Path Setup
import os
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.abo_loader import (
    ABOCatalogLoader,
    download_file,
    extract_tar,
    split_by_sku,
    save_manifests_and_stats,
    ABO_LISTINGS_URL,
    ABO_IMAGES_SMALL_URL,
)
from src.data.augmentations import apply_multimodal_augmentations, get_image_augmentations, get_text_augmentations
from src.data.pair_sampler import PairSampler
from src.data.dataset import FusionMatchDataset, build_dataloaders

print(f"Project root: {PROJECT_ROOT}")

## 1. Download & Extract Dataset (Idempotent)
Downloads `abo-listings.tar` (~85MB) and `abo-images-small.tar` (~3.25GB) from official Amazon S3 archives.

In [ ]:
# Cell 2: Download and extraction
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

listings_tar = raw_dir / "abo-listings.tar"
images_tar = raw_dir / "abo-images-small.tar"

listings_extracted = raw_dir / "abo-listings"
images_extracted = raw_dir / "abo-images-small"

# 1. Download listings archive
if not listings_extracted.exists():
    download_file(ABO_LISTINGS_URL, listings_tar)
    extract_tar(listings_tar, listings_extracted)

# 2. Download images archive
if not images_extracted.exists():
    download_file(ABO_IMAGES_SMALL_URL, images_tar)
    extract_tar(images_tar, images_extracted)

print("Extraction complete:")
print(f"  Listings: {listings_extracted}")
print(f"  Images:   {images_extracted}")

## 2. Manifest Generation & Stratified Sampling
Parses product metadata, filters invalid rows, and stratifies down to target ~11,000 SKUs.

In [ ]:
# Cell 3: Load and Stratify Catalog
loader = ABOCatalogLoader(
    images_root=images_extracted,
    listings_root=listings_extracted,
    max_skus=11000,
    min_images_per_sku=1,
    seed=42,
)

manifest_df = loader.load_manifest()
print(f"Total manifest rows: {len(manifest_df)}")
print(f"Unique SKUs: {manifest_df['sku_id'].nunique()}")
print(f"Unique categories: {manifest_df['category'].nunique()}")

### Sample Parsed Records

In [ ]:
# Cell 4: View Sample Records
manifest_df.head(5)[["sku_id", "title", "brand", "category", "image_path"]]

## 3. Exploratory Data Analysis (EDA)
Inspects category balance and the distribution of images per SKU (multi-angle photography).

In [ ]:
# Cell 5: Category Distribution & Multi-Angle Distribution Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top categories
top_cats = manifest_df["category"].value_counts().head(15)
top_cats.plot(kind="barh", ax=axes[0], color="#3b82f6", edgecolor="black")
axes[0].set_title("Top 15 Product Categories", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Image Count")
axes[0].invert_yaxis()

# Images per SKU distribution
imgs_per_sku = manifest_df.groupby("sku_id")["image_path"].count()
imgs_per_sku.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#10b981", edgecolor="black")
axes[1].set_title("Images per SKU Distribution (Multi-Angle)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Images per SKU")
axes[1].set_ylabel("Number of SKUs")

plt.tight_layout()
plt.show()

## 4. SKU-Level Train / Val / Test Partitioning
Partitions the dataset strictly by **SKU** (80% Train, 10% Val, 10% Test) to prevent data leakage.

In [ ]:
# Cell 6: Split by SKU and verify zero leakage
train_df, val_df, test_df = split_by_sku(manifest_df, seed=42, ratios=(0.8, 0.1, 0.1))

# Disjointness verification
train_skus = set(train_df["sku_id"])
val_skus = set(val_df["sku_id"])
test_skus = set(test_df["sku_id"])

print("=== Leakage Verification ===")
print(f"Train ∩ Val overlap:  {len(train_skus & val_skus)} (Must be 0)")
print(f"Train ∩ Test overlap: {len(train_skus & test_skus)} (Must be 0)")
print(f"Val ∩ Test overlap:   {len(val_skus & test_skus)} (Must be 0)")
assert len(train_skus & val_skus) == 0 and len(train_skus & test_skus) == 0 and len(val_skus & test_skus) == 0

# Save processed manifests
save_manifests_and_stats(train_df, val_df, test_df, output_dir=PROJECT_ROOT / "data" / "processed")

## 5. Visualizing Positive & Negative Pairs
Renders real multi-angle positive pairs, augmented duplicates, and in-category hard negatives side-by-side.

In [ ]:
# Cell 7: Pair Sampling Demonstration
sampler = PairSampler(train_df, seed=42)

if sampler.multi_image_skus:
    sample_sku = sampler.multi_image_skus[0]
    anc, pos, is_multi = sampler.sample_positive_pair(sample_sku)
    neg = sampler.sample_hard_negative(sample_sku)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    img_anc = Image.open(anc["image_path"]).convert("RGB")
    axes[0].imshow(img_anc)
    axes[0].set_title(f"Anchor: {anc['sku_id']}\n{anc['title'][:35]}...", fontsize=10)
    axes[0].axis("off")

    img_pos = Image.open(pos["image_path"]).convert("RGB")
    axes[1].imshow(img_pos)
    axes[1].set_title(f"Positive (Multi-Angle): {pos['sku_id']}\n{pos['title'][:35]}...", fontsize=10)
    axes[1].axis("off")

    img_neg = Image.open(neg["image_path"]).convert("RGB")
    axes[2].imshow(img_neg)
    axes[2].set_title(f"Hard Negative: {neg['sku_id']}\n{neg['title'][:35]}...", fontsize=10)
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

## 6. PyTorch Dataset & DataLoader Verification
Verifies batch shape compatibility with SigLIP input specs.

In [ ]:
# Cell 8: DataLoader Smoke Test
train_loader, val_loader, test_loader = build_dataloaders(
    train_manifest=PROJECT_ROOT / "data" / "processed" / "manifest_train.csv",
    val_manifest=PROJECT_ROOT / "data" / "processed" / "manifest_val.csv",
    test_manifest=PROJECT_ROOT / "data" / "processed" / "manifest_test.csv",
    batch_size=16,
    num_workers=0,
)

batch = next(iter(train_loader))
print("Batch keys:", list(batch.keys()))
print(f"Anchor pixel values shape:   {batch['anchor_pixel_values'].shape}")
print(f"Positive pixel values shape: {batch['positive_pixel_values'].shape}")
print(f"Anchor input IDs shape:      {batch['anchor_input_ids'].shape}")
print(f"Anchor quality proxies:      q_v={batch['anchor_q_v'].shape}, q_t={batch['anchor_q_t'].shape}")